In [1]:
!pip install -q langchain langchain-cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 23.9 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

os.environ["COHERE_API_KEY"] = getpass("Enter your Cohere API key: ")

Enter your Cohere API key: ··········


In [3]:
from langchain_cohere import ChatCohere

llm = ChatCohere(model="command-a-03-2025")

response = llm.invoke("Explain Generative AI in simple terms.")

print(response.content)

Generative AI is a type of artificial intelligence that can create new content, such as text, images, music, or even videos, that didn’t exist before. Think of it like a creative assistant that uses patterns it has learned from lots of data to produce something original.

Here’s a simple analogy: Imagine you’ve read hundreds of stories about cats. If someone asks you to write a new story about a cat, you can use what you’ve learned to create a unique tale. Generative AI does something similar—it “reads” or analyzes a large amount of data (like text, pictures, or music) and then uses that knowledge to generate new, similar content.

Examples of generative AI include:
- **Chatbots** like ChatGPT that write essays, answer questions, or tell jokes.
- **Image generators** like DALL·E or MidJourney that create pictures from text descriptions.
- **Music composers** that produce new songs in a specific style.

In short, generative AI is like a smart, creative tool that can make new things base

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms with one real-world example."
)

chain = prompt | llm

response = chain.invoke({"topic": "Retrieval Augmented Generation"})

print(response.content)

**Retrieval Augmented Generation (RAG)** is a technique that combines the strengths of two approaches in artificial intelligence: **retrieving** relevant information from a large database and **generating** coherent, contextually appropriate text. In simpler terms, it’s like having a smart assistant that first looks up the most useful facts or details (retrieval) and then uses those to write a well-informed answer (generation).

### How it works:
1. **Retrieval**: The system searches a large dataset (like a database, documents, or the internet) to find the most relevant pieces of information for a given question or topic.
2. **Generation**: It then uses a language model (like GPT) to craft a response based on the retrieved information, ensuring the answer is accurate, detailed, and coherent.

### Real-world example: **Customer Support Chatbot**
Imagine a customer asks a chatbot, *"What are the return policies for electronics purchased during the holiday sale?"*

1. **Retrieval**: The c

In [5]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_cohere import ChatCohere

# Step 1: Connect to Cohere
llm = ChatCohere(model="command-a-03-2025")

# Step 2: Create a tool
@tool
def calculate_average(marks: list[int]) -> float:
    """Calculate the average of a list of student marks."""
    return sum(marks) / len(marks) if marks else 0.0

# Step 3: Create the agent
agent = create_agent(
    model=llm,
    tools=[calculate_average],
    system_prompt=(
        "You are a student marks analysis assistant. "
        "Use the calculate_average tool when asked to calculate "
        "the average of student marks."
    )
)

# Step 4: Give the agent a task
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Calculate the average of marks 80, 75, 90 and 85."
        }
    ]
})

# Step 5: Display the final answer
print(result["messages"][-1].content)

The average of the marks 80, 75, 90 and 85 is 82.5.


In [6]:
!pip install -q -U langgraph langchain-core langchain-cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.6/259.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.5/862.5 kB 42.4 MB/s eta 0:00:00


In [7]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from langchain_cohere import ChatCohere

# 1. Define the state
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 2. Create the Cohere model
llm = ChatCohere(model="command-a-03-2025")

# 3. Define the graph node
def chatbot(state: State):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

# 4. Build the graph
builder = StateGraph(State)

builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# 5. Compile the graph
graph = builder.compile()

# 6. Run the graph
result = graph.invoke({
    "messages": [
        HumanMessage(content="Explain machine learning simply.")
    ]
})

print(result["messages"][-1].content)

Sure! Here's a simple explanation of machine learning:

**Machine learning** is a way of teaching computers to learn from data and make decisions or predictions without being explicitly programmed for every specific task. Think of it like teaching a child to recognize cats by showing them many pictures of cats. Over time, the child learns what a cat looks like and can identify one on their own.

In machine learning, the "child" is a computer program, and the "pictures" are data (like numbers, images, or text). The program uses this data to find patterns and make decisions or predictions. For example:

- **Email spam filter**: The program learns from many emails labeled as "spam" or "not spam" to decide which emails to filter out.
- **Recommendation systems**: Netflix or Spotify learns from what you watch or listen to and suggests new shows or songs you might like.
- **Self-driving cars**: The car learns from thousands of hours of driving data to recognize roads, signs, and obstacles.



In [8]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_cohere import ChatCohere
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# STEP 1: Define the state
class State(TypedDict):
    messages: Annotated[list, add_messages]

# STEP 2: Connect to Cohere
llm = ChatCohere(model="command-a-03-2025", temperature=0)

# STEP 3: Create a tool
@tool
def calculate_average(marks: list[int]) -> float:
    """Calculate the average of student marks."""
    if not marks:
        return 0.0
    return sum(marks) / len(marks)

tools = [calculate_average]

# STEP 4: Allow the LLM to request tool calls
llm_with_tools = llm.bind_tools(tools)

# STEP 5: Define the LLM node
def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# STEP 6: Build the graph
builder = StateGraph(State)

builder.add_node("chatbot", chatbot)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "chatbot")

# Route to the tool if the model requests one
builder.add_conditional_edges(
    "chatbot",
    tools_condition
)

# After tool execution, return to the LLM
builder.add_edge("tools", "chatbot")

# STEP 7: Compile
graph = builder.compile()

# STEP 8: Run the agent
result = graph.invoke({
    "messages": [
        HumanMessage(
            content="Calculate the average of marks 80, 75, 90 and 85."
        )
    ]
})

print(result["messages"][-1].content)

The average of the marks 80, 75, 90 and 85 is 82.5.


In [ ]:
#!pip install -q gradio

import gradio as gr
from langchain_core.messages import HumanMessage, AIMessage

def chat_with_agent(message, history):
    # Prepare the query using the user input
    inputs = {
        "messages": [HumanMessage(content=message)]
    }

    # Invoke the compiled graph
    result = graph.invoke(inputs)

    # The last message from the run will be our chatbot output
    final_response = result["messages"][-1].content
    return final_response

# Create and launch the Gradio interface
demo = gr.ChatInterface(
    fn=chat_with_agent,
    title="AI Marks Analyst Agent",
    description="Ask me to calculate average marks or analyze your scores!"
)

# Launch with inline display within Colab
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://449ad457b58ee4b7e5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
